In [2]:
from pathlib import Path
import sqlite3

project_root = Path.cwd()

print("Current working directory:")
print(project_root)

Current working directory:
C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics\notebooks


In [3]:
from pathlib import Path
import sqlite3

notebook_dir = Path.cwd()
project_root = notebook_dir.parent

print("Notebook directory:")
print(notebook_dir)

print("\nProject root:")
print(project_root)

Notebook directory:
C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics\notebooks

Project root:
C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics


In [7]:
from pathlib import Path
import sqlite3

# Notebook is inside /notebooks, so project root is one level up
project_root = Path.cwd().parent

db_path = project_root / "data" / "database" / "superstore_commercial_analytics.db"
sql_script_path = project_root / "sql" / "01_raw_staging.sql"

# Make sure database folder exists
db_path.parent.mkdir(parents=True, exist_ok=True)

print("Database path:", db_path)
print("SQL script path:", sql_script_path)

# Read SQL script
sql_script = sql_script_path.read_text(encoding="utf-8")

# Create database and execute table creation script
conn = sqlite3.connect(db_path)

try:
    conn.executescript(sql_script)
    conn.commit()
    print("Database and tables created successfully.")
finally:
    conn.close()

Database path: C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics\data\database\superstore_commercial_analytics.db
SQL script path: C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics\sql\01_raw_staging.sql
Database and tables created successfully.


In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

# Notebook is inside /notebooks, so project root is one level up
project_root = Path.cwd().parent

raw_path = project_root / "data" / "raw" / "Sample - Superstore.csv"
processed_path = project_root / "data" / "processed" / "stg_superstore_sales.csv"

# Make sure processed folder exists
processed_path.parent.mkdir(parents=True, exist_ok=True)

print("Raw path:", raw_path)
print("Processed output path:", processed_path)

# Load raw file
df = pd.read_csv(raw_path, encoding="latin-1")

# Rename columns
rename_map = {
    "Row ID": "row_id",
    "Order ID": "order_id",
    "Order Date": "order_date",
    "Ship Date": "ship_date",
    "Ship Mode": "ship_mode",
    "Customer ID": "customer_id",
    "Customer Name": "customer_name",
    "Segment": "segment",
    "Country": "country",
    "City": "city",
    "State": "state",
    "Postal Code": "postal_code",
    "Region": "region",
    "Product ID": "product_id",
    "Category": "category",
    "Sub-Category": "sub_category",
    "Product Name": "product_name",
    "Sales": "sales",
    "Quantity": "quantity",
    "Discount": "discount",
    "Profit": "profit"
}

df = df.rename(columns=rename_map)

# Parse dates
df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"] = pd.to_datetime(df["ship_date"])

# Preserve postal code as label/text
df["postal_code"] = df["postal_code"].astype(str)

# Derived helper fields
df["order_year"] = df["order_date"].dt.year
df["order_month"] = df["order_date"].dt.to_period("M").astype(str)
df["ship_days"] = (df["ship_date"] - df["order_date"]).dt.days
df["profit_flag"] = np.where(df["profit"] < 0, "Loss-Making", "Profitable")

discount_bins = [-0.01, 0.0, 0.15, 0.30, 1.0]
discount_labels = [
    "No Discount",
    "Low Discount",
    "Moderate Discount",
    "High Discount"
]

df["discount_tier"] = pd.cut(
    df["discount"],
    bins=discount_bins,
    labels=discount_labels
)

# Save staged file
df.to_csv(processed_path, index=False)

print("\nStaged file created successfully.")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
print("Saved to:", processed_path)

df.head()

Raw path: C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics\data\raw\Sample - Superstore.csv
Processed output path: C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics\data\processed\stg_superstore_sales.csv

Staged file created successfully.
Rows: 9,994
Columns: 26
Saved to: C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics\data\processed\stg_superstore_sales.csv


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,product_name,sales,quantity,discount,profit,order_year,order_month,ship_days,profit_flag,discount_tier
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016,2016-11,3,Profitable,No Discount
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016,2016-11,3,Profitable,No Discount
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016,2016-06,4,Profitable,No Discount
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015,2015-10,7,Loss-Making,High Discount
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015,2015-10,7,Profitable,Moderate Discount


In [10]:
from pathlib import Path
import sqlite3
import pandas as pd

# Notebook is inside /notebooks, so project root is one level up
project_root = Path.cwd().parent

db_path = project_root / "data" / "database" / "superstore_commercial_analytics.db"
staged_csv_path = project_root / "data" / "processed" / "stg_superstore_sales.csv"

print("Database path:", db_path)
print("Staged CSV path:", staged_csv_path)

# Load staged CSV
df = pd.read_csv(staged_csv_path)

print("\nStaged CSV loaded.")
print(f"Rows in CSV: {len(df):,}")
print(f"Columns in CSV: {df.shape[1]:,}")
print(df.head())

# Connect to SQLite
conn = sqlite3.connect(db_path)

try:
    # Load dataframe into existing SQL table
    df.to_sql(
        "raw_superstore_sales",
        conn,
        if_exists="replace",
        index=False
    )

    # Validate row count from SQL
    sql_count = pd.read_sql_query(
        "SELECT COUNT(*) AS row_count FROM raw_superstore_sales;",
        conn
    )

    print("\nSQL row count validation:")
    print(sql_count)

finally:
    conn.close()

Database path: C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics\data\database\superstore_commercial_analytics.db
Staged CSV path: C:\Projects\AnalyticsPortfolio\superstore-commercial-analytics\data\processed\stg_superstore_sales.csv

Staged CSV loaded.
Rows in CSV: 9,994
Columns in CSV: 26
   row_id        order_id  order_date   ship_date       ship_mode customer_id  \
0       1  CA-2016-152156  2016-11-08  2016-11-11    Second Class    CG-12520   
1       2  CA-2016-152156  2016-11-08  2016-11-11    Second Class    CG-12520   
2       3  CA-2016-138688  2016-06-12  2016-06-16    Second Class    DV-13045   
3       4  US-2015-108966  2015-10-11  2015-10-18  Standard Class    SO-20335   
4       5  US-2015-108966  2015-10-11  2015-10-18  Standard Class    SO-20335   

     customer_name    segment        country             city  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...  

In [12]:
from pathlib import Path
import sqlite3
import pandas as pd

project_root = Path.cwd().parent
db_path = project_root / "data" / "database" / "superstore_commercial_analytics.db"

def run_sql(query):
    """
    Run a SQL query against the Superstore SQLite database
    and return the result as a pandas DataFrame.
    """
    conn = sqlite3.connect(db_path)
    try:
        result = pd.read_sql_query(query, conn)
    finally:
        conn.close()
    return result

In [13]:
run_sql("""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT row_id) AS distinct_row_ids,
    COUNT(DISTINCT order_id) AS distinct_orders,
    COUNT(DISTINCT customer_id) AS distinct_customers,
    COUNT(DISTINCT product_id) AS distinct_products
FROM raw_superstore_sales;
""")

,row_count,distinct_row_ids,distinct_orders,distinct_customers,distinct_products
0,9994,9994,5009,793,1862


# Phase 2 QA Result — QA 001 Core Count Validation

QA 001 passed.

Results:
- Total rows: 9,994
- Distinct row IDs: 9,994
- Distinct orders: 5,009
- Distinct customers: 793
- Distinct products: 1,862

Interpretation:
The staged table contains 9,994 product line-item records across 5,009 distinct orders. The row_id field is unique and can be treated as the row-level key. Customer and product identifiers are available for aggregation, subject to later key-consistency checks.

Grain implication:
One row represents one product line item within an order, not one complete order.

In [15]:
run_sql("""
SELECT
    row_id,
    COUNT(*) AS duplicate_count
FROM raw_superstore_sales
GROUP BY row_id
HAVING COUNT(*) > 1;
""")

,row_id,duplicate_count


# Phase 2 QA Result — QA 002 Duplicate Row ID Check

QA 002 passed.

Result:
- Duplicate row_id records returned: 0

Interpretation:
No row_id appears more than once in the staged SQLite table. This supports the decision to treat row_id as the row-level key.

Analytical implication:
Sales, profit, quantity and discount values are not inflated by duplicate row_id records.

In [16]:
run_sql("""
SELECT
    order_id,
    COUNT(*) AS line_item_count
FROM raw_superstore_sales
GROUP BY order_id
ORDER BY line_item_count DESC
LIMIT 20;
""")

,order_id,line_item_count
0,CA-2017-100111,14
1,CA-2017-157987,12
2,US-2016-108504,11
3,CA-2016-165330,11
4,US-2015-126977,10
5,CA-2016-105732,10
6,CA-2015-131338,10
7,US-2016-114013,9
8,US-2015-163433,9
9,CA-2017-140949,9


# Phase 2 QA Result — QA 003 Order-Level Grain Check

QA 003 confirmed the row grain.

Result:
The largest observed orders contain multiple product line items. The maximum line-item count observed in the top 20 output was 14 for order CA-2017-100111. Other orders also contained 8–12 line items.

Interpretation:
The staged Superstore table is not one row per order. One row represents one product line item within an order.

Analytical implication:
COUNT(*) should be interpreted as line-item count, not order count. Order-level analysis must use COUNT(DISTINCT order_id) or group by order_id. Customer-level and product-level analysis must use the appropriate distinct IDs.

In [17]:
run_sql("""
SELECT
    MIN(order_date) AS min_order_date,
    MAX(order_date) AS max_order_date,
    MIN(ship_date) AS min_ship_date,
    MAX(ship_date) AS max_ship_date
FROM raw_superstore_sales;
""")

,min_order_date,max_order_date,min_ship_date,max_ship_date
0,2014-01-03,2017-12-30,2014-01-07,2018-01-05


# Phase 2 QA Result — QA 004 Date Range Validation

QA 004 passed.

Results:
- Minimum Order Date: 2014-01-03
- Maximum Order Date: 2017-12-30
- Minimum Ship Date: 2014-01-07
- Maximum Ship Date: 2018-01-05

Interpretation:
The primary sales/profit reporting period covers orders from 2014 through 2017. Ship Date extends into early 2018 because late-2017 orders can ship after the order year ends.

Decision:
Use Order Date as the default business date for sales, profit, margin and year-over-year analysis. Use Ship Date only for fulfillment or shipping-timing questions.

Analytical implication:
Annual sales and profit trends should be grouped by Order Date, not Ship Date.

In [18]:
run_sql("""
SELECT
    COUNT(*) AS ship_before_order_count
FROM raw_superstore_sales
WHERE ship_date < order_date;
""")

,ship_before_order_count
0,0


# Phase 2 QA Result — QA 005 Ship Date Before Order Date Check

QA 005 passed.

Result:
- Records where Ship Date is earlier than Order Date: 0

Interpretation:
No impossible shipping-date sequences were found. The date fields are internally consistent for basic order-to-ship timing checks.

Decision:
The derived ship_days field can be used as a valid helper field if fulfillment timing is reviewed later. However, Order Date remains the default business date for sales, profit, margin and year-over-year analysis.

Analytical implication:
Date sequencing does not create a data-quality issue for sales/profit trend analysis.

In [19]:
run_sql("""
SELECT
    order_year,
    COUNT(*) AS row_count,
    COUNT(DISTINCT order_id) AS order_count,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(SUM(profit) / SUM(sales), 4) AS weighted_profit_margin
FROM raw_superstore_sales
GROUP BY order_year
ORDER BY order_year;
""")

,order_year,row_count,order_count,total_sales,total_profit,weighted_profit_margin
0,2014,1993,969,484247.50,49543.97,0.1023
1,2015,2102,1038,470532.51,61618.60,0.1310
2,2016,2587,1315,609205.60,81795.17,0.1343
3,2017,3312,1687,733215.26,93439.27,0.1274


# Phase 2 QA Result — QA 006 Annual Coverage Check

QA 006 passed.

Results:
2014:
- Rows: 1,993
- Orders: 969
- Sales: 484,247.50
- Profit: 49,543.97
- Weighted Profit Margin: 10.23%

2015:
- Rows: 2,102
- Orders: 1,038
- Sales: 470,532.51
- Profit: 61,618.60
- Weighted Profit Margin: 13.10%

2016:
- Rows: 2,587
- Orders: 1,315
- Sales: 609,205.60
- Profit: 81,795.17
- Weighted Profit Margin: 13.43%

2017:
- Rows: 3,312
- Orders: 1,687
- Sales: 733,215.26
- Profit: 93,439.27
- Weighted Profit Margin: 12.74%

Interpretation:
The dataset has usable order-date coverage across 2014, 2015, 2016 and 2017. Year-over-year analysis is structurally feasible.

Caution:
Row count and order count increase over time. Sales and profit growth may reflect increased transaction volume, not necessarily improved profitability or efficiency.

Decision:
YoY analysis remains a valid fallback or supporting branch after Phase 3 metric development. Use Order Date and weighted profit margin for annual trend analysis.

In [20]:
run_sql("""
SELECT
    SUM(CASE WHEN row_id IS NULL THEN 1 ELSE 0 END) AS missing_row_id,
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS missing_order_id,
    SUM(CASE WHEN order_date IS NULL THEN 1 ELSE 0 END) AS missing_order_date,
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS missing_customer_id,
    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS missing_product_id,
    SUM(CASE WHEN sales IS NULL THEN 1 ELSE 0 END) AS missing_sales,
    SUM(CASE WHEN profit IS NULL THEN 1 ELSE 0 END) AS missing_profit,
    SUM(CASE WHEN discount IS NULL THEN 1 ELSE 0 END) AS missing_discount
FROM raw_superstore_sales;
""")

,missing_row_id,missing_order_id,missing_order_date,missing_customer_id,missing_product_id,missing_sales,missing_profit,missing_discount
0,0,0,0,0,0,0,0,0


# Phase 2 QA Result — QA 007 Missingness Check for Key Fields

QA 007 passed.

Results:
- Missing row_id: 0
- Missing order_id: 0
- Missing order_date: 0
- Missing customer_id: 0
- Missing product_id: 0
- Missing sales: 0
- Missing profit: 0
- Missing discount: 0

Interpretation:
No missing values were found in the critical fields needed for row-level validation, order-level aggregation, customer analysis, product analysis, sales/profit metrics, discount analysis or date-based trends.

Decision:
The staged table is complete enough to proceed with category, segment, region, discount, and customer/product key consistency checks.

Analytical implication:
Missingness in critical fields does not create a data-quality limitation for the planned commercial profitability analysis.

In [21]:
run_sql("""
SELECT
    category,
    COUNT(*) AS row_count,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM raw_superstore_sales
GROUP BY category
ORDER BY category;
""")

,category,row_count,total_sales,total_profit
0,Furniture,2121,741999.80,18451.27
1,Office Supplies,6026,719047.03,122490.80
2,Technology,1847,836154.03,145454.95


# Phase 2 QA Result — QA 008 Category Coverage Check

QA 008 passed.

Results:
- Furniture: 2,121 rows, 741,999.80 sales, 18,451.27 profit
- Office Supplies: 6,026 rows, 719,047.03 sales, 122,490.80 profit
- Technology: 1,847 rows, 836,154.03 sales, 145,454.95 profit

Interpretation:
The staged table contains three clean major product categories: Furniture, Office Supplies and Technology. Category labels are consistent and usable for product-mix analysis.

Early QA observation:
Furniture has high sales but substantially lower profit than Office Supplies and Technology. This may become stakeholder-relevant during Phase 3, but it should not be finalized as a finding until sub-category, discount and margin checks are completed.

Decision:
Category-level product-mix analysis is feasible and should be included in Phase 3.

In [22]:
run_sql("""
SELECT
    category,
    sub_category,
    COUNT(*) AS row_count,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM raw_superstore_sales
GROUP BY category, sub_category
ORDER BY category, sub_category;
""")

,category,sub_category,row_count,total_sales,total_profit
0,Furniture,Bookcases,228,114880.00,-3472.56
1,Furniture,Chairs,617,328449.10,26590.17
2,Furniture,Furnishings,957,91705.16,13059.14
3,Furniture,Tables,319,206965.53,-17725.48
4,Office Supplies,Appliances,466,107532.16,18138.01
5,Office Supplies,Art,796,27118.79,6527.79
6,Office Supplies,Binders,1523,203412.73,30221.76
7,Office Supplies,Envelopes,254,16476.40,6964.18
8,Office Supplies,Fasteners,217,3024.28,949.52
9,Office Supplies,Labels,364,12486.31,5546.25


# Phase 2 QA Result — QA 009 Sub-Category Coverage Check

QA 009 passed.

Results:
The staged table contains 17 clean sub-categories under 3 major product categories.

Furniture sub-categories:
- Bookcases: 228 rows, 114,880.00 sales, -3,472.56 profit
- Chairs: 617 rows, 328,449.10 sales, 26,590.17 profit
- Furnishings: 957 rows, 91,705.16 sales, 13,059.14 profit
- Tables: 319 rows, 206,965.53 sales, -17,725.48 profit

Office Supplies sub-categories:
- Appliances: 466 rows, 107,532.16 sales, 18,138.01 profit
- Art: 796 rows, 27,118.79 sales, 6,527.79 profit
- Binders: 1,523 rows, 203,412.73 sales, 30,221.76 profit
- Envelopes: 254 rows, 16,476.40 sales, 6,964.18 profit
- Fasteners: 217 rows, 3,024.28 sales, 949.52 profit
- Labels: 364 rows, 12,486.31 sales, 5,546.25 profit
- Paper: 1,370 rows, 78,479.21 sales, 34,053.57 profit
- Storage: 846 rows, 223,843.61 sales, 21,278.83 profit
- Supplies: 190 rows, 46,673.54 sales, -1,189.10 profit

Technology sub-categories:
- Accessories: 775 rows, 167,380.32 sales, 41,936.64 profit
- Copiers: 68 rows, 149,528.03 sales, 55,617.82 profit
- Machines: 115 rows, 189,238.63 sales, 3,384.76 profit
- Phones: 889 rows, 330,007.05 sales, 44,515.73 profit

Interpretation:
Sub-category labels are clean and usable for product-mix analysis. Furniture weakness appears concentrated in Tables and Bookcases rather than the entire Furniture category. Technology shows strong profit contribution from Copiers, Phones and Accessories, while Machines have high sales but comparatively low profit. Office Supplies is broadly profitable, with Supplies showing a small negative profit total.

Decision:
Sub-category analysis is feasible and should be included in Phase 3. Sub-category should be a primary product-mix storytelling level because it is detailed enough for commercial review while avoiding product-name ambiguity.

In [23]:
run_sql("""
SELECT
    segment,
    COUNT(*) AS row_count,
    COUNT(DISTINCT customer_id) AS customer_count,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM raw_superstore_sales
GROUP BY segment
ORDER BY segment;
""")

,segment,row_count,customer_count,total_sales,total_profit
0,Consumer,5191,409,1161401.34,134119.21
1,Corporate,3020,236,706146.37,91979.13
2,Home Office,1783,148,429653.15,60298.68


# Phase 2 QA Result — QA 010 Segment Coverage Check

QA 010 passed.

Results:
- Consumer: 5,191 rows, 409 customers, 1,161,401.34 sales, 134,119.21 profit
- Corporate: 3,020 rows, 236 customers, 706,146.37 sales, 91,979.13 profit
- Home Office: 1,783 rows, 148 customers, 429,653.15 sales, 60,298.68 profit

Interpretation:
The staged table contains three clean customer segments: Consumer, Corporate and Home Office. All three segments have enough records and customers to support segment-level analysis.

Early QA observation:
Consumer is the largest segment by rows, customers, sales and profit. Corporate is second, and Home Office is smallest. This should not be treated as a final finding yet because segment-level margin, discount exposure, product mix and loss-making sales share still need to be tested.

Decision:
Segment-level analysis is feasible and should be included as a supporting commercial dimension in Phase 3. It should not dominate the project unless segment profitability or discount patterns reveal stakeholder-relevant variation.

In [24]:
run_sql("""
SELECT
    region,
    COUNT(*) AS row_count,
    COUNT(DISTINCT order_id) AS order_count,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM raw_superstore_sales
GROUP BY region
ORDER BY region;
""")

,region,row_count,order_count,total_sales,total_profit
0,Central,2323,1175,501239.89,39706.36
1,East,2848,1401,678781.24,91522.78
2,South,1620,822,391721.91,46749.43
3,West,3203,1611,725457.82,108418.45


# Phase 2 QA Result — QA 011 Region Coverage Check

QA 011 passed.

Results:
- Central: 2,323 rows, 1,175 orders, 501,239.89 sales, 39,706.36 profit
- East: 2,848 rows, 1,401 orders, 678,781.24 sales, 91,522.78 profit
- South: 1,620 rows, 822 orders, 391,721.91 sales, 46,749.43 profit
- West: 3,203 rows, 1,611 orders, 725,457.82 sales, 108,418.45 profit

Interpretation:
The staged table contains four clean regional labels: Central, East, South and West. All regions have enough rows and orders to support regional analysis.

Early QA observation:
West has the highest sales and profit, followed by East. Central has higher sales than South but lower profit, suggesting that regional margin analysis may be more useful than sales ranking alone.

Decision:
Regional analysis is feasible and should be included in Phase 3. Region-level analysis should compare both total contribution and weighted profit margin, not just sales volume.

In [25]:
run_sql("""
SELECT
    ship_mode,
    COUNT(*) AS row_count,
    COUNT(DISTINCT order_id) AS order_count
FROM raw_superstore_sales
GROUP BY ship_mode
ORDER BY ship_mode;
""")

,ship_mode,row_count,order_count
0,First Class,1538,787
1,Same Day,543,264
2,Second Class,1945,964
3,Standard Class,5968,2994


# Phase 2 QA Result — QA 012 Ship Mode Coverage Check

QA 012 passed.

Results:
- First Class: 1,538 rows, 787 orders
- Same Day: 543 rows, 264 orders
- Second Class: 1,945 rows, 964 orders
- Standard Class: 5,968 rows, 2,994 orders

Interpretation:
The staged table contains four clean ship mode labels: First Class, Same Day, Second Class and Standard Class. No obvious category consistency issues were found.

Decision:
Ship mode is valid for supporting analysis or dashboard filtering, but it should not become a core project focus unless later analysis shows a clear connection to profitability or revenue leakage.

Analytical implication:
Do not expand this mini project into a logistics analysis. Keep ship mode as optional context.

In [26]:
run_sql("""
SELECT
    MIN(sales) AS min_sales,
    MAX(sales) AS max_sales,
    MIN(quantity) AS min_quantity,
    MAX(quantity) AS max_quantity,
    MIN(discount) AS min_discount,
    MAX(discount) AS max_discount,
    MIN(profit) AS min_profit,
    MAX(profit) AS max_profit,
    MIN(ship_days) AS min_ship_days,
    MAX(ship_days) AS max_ship_days
FROM raw_superstore_sales;
""")

,min_sales,max_sales,min_quantity,max_quantity,min_discount,max_discount,min_profit,max_profit,min_ship_days,max_ship_days
0,0.444,22638.48,1,14,0.0,0.8,-6599.978,8399.976,0,7


# Phase 2 QA Result — QA 013 Numeric Sanity Check

QA 013 passed.

Results:
- Minimum Sales: 0.444
- Maximum Sales: 22,638.48
- Minimum Quantity: 1
- Maximum Quantity: 14
- Minimum Discount: 0.0
- Maximum Discount: 0.8
- Minimum Profit: -6,599.978
- Maximum Profit: 8,399.976
- Minimum Ship Days: 0
- Maximum Ship Days: 7

Interpretation:
The core numeric fields are within usable ranges for the sample dataset. Sales and quantity are positive. Discount ranges from 0% to 80%. Profit includes both large losses and large gains. Ship Days is non-negative and ranges from 0 to 7 days.

Decision:
The numeric fields are valid enough for Phase 3 metric development and analysis.

Analytical caution:
Large positive and negative profit values may influence category, sub-category, product and customer-level profitability metrics. Profit outliers should be reviewed or acknowledged during analysis.

In [27]:
run_sql("""
SELECT
    COUNT(*) AS sales_lte_zero_count
FROM raw_superstore_sales
WHERE sales <= 0;
""")

,sales_lte_zero_count
0,0


# Phase 2 QA Result — QA 014 Sales Less Than or Equal to Zero

QA 014 passed.

Result:
- Sales less than or equal to zero: 0 rows

Interpretation:
No zero or negative sales records were found. This supports using Sales as the denominator for weighted profit margin and sales-share calculations.

Decision:
Sales is valid for total sales, weighted profit margin, sales share, category/region/segment contribution, and year-over-year sales analysis.

In [28]:
run_sql("""
SELECT
    COUNT(*) AS negative_profit_row_count
FROM raw_superstore_sales
WHERE profit < 0;
""")

,negative_profit_row_count
0,1871


# Phase 2 QA Result — QA 015 Negative Profit Count

QA 015 passed.

Result:
- Negative profit line items: 1,871

Interpretation:
The staged dataset contains 1,871 loss-making product line items out of 9,994 total rows, or approximately 18.7% of line items. Negative profit rows are analytically meaningful and should not be treated as automatic data errors.

Decision:
Negative profit analysis is feasible and should be included in Phase 3. Potential metrics include loss-making line-item count, loss-making row share, loss-making sales, loss-making profit impact, and negative profit patterns by category, sub-category, region and discount tier.

Caution:
Negative profit records show observed losses, but they do not prove why the losses occurred. Do not claim that discounts caused losses without stronger evidence.

In [29]:
run_sql("""
SELECT
    discount,
    COUNT(*) AS row_count,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM raw_superstore_sales
GROUP BY discount
ORDER BY discount;
""")

,discount,row_count,total_sales,total_profit
0,0.00,4798,1087908.47,320987.60
1,0.10,94,54369.35,9029.18
2,0.15,52,27558.52,1418.99
3,0.20,3657,764594.37,90337.31
4,0.30,227,103226.65,-10369.28
5,0.32,27,14493.46,-2391.14
6,0.40,206,116417.78,-23057.05
7,0.45,11,5484.97,-2493.11
8,0.50,66,58918.54,-20506.43
9,0.60,138,6644.70,-5944.66


# Phase 2 QA Result — QA 016 Discount Value Distribution

QA 016 passed.

Results:
- 0% discount: 4,798 rows, 1,087,908.47 sales, 320,987.60 profit
- 10% discount: 94 rows, 54,369.35 sales, 9,029.18 profit
- 15% discount: 52 rows, 27,558.52 sales, 1,418.99 profit
- 20% discount: 3,657 rows, 764,594.37 sales, 90,337.31 profit
- 30% discount: 227 rows, 103,226.65 sales, -10,369.28 profit
- 32% discount: 27 rows, 14,493.46 sales, -2,391.14 profit
- 40% discount: 206 rows, 116,417.78 sales, -23,057.05 profit
- 45% discount: 11 rows, 5,484.97 sales, -2,493.11 profit
- 50% discount: 66 rows, 58,918.54 sales, -20,506.43 profit
- 60% discount: 138 rows, 6,644.70 sales, -5,944.66 profit
- 70% discount: 418 rows, 40,620.28 sales, -40,075.36 profit
- 80% discount: 300 rows, 16,963.76 sales, -30,539.04 profit

Interpretation:
Discount values are clean and intentionally banded. The data supports discount-level analysis. Observed discounts of 0%, 10%, 15% and 20% show positive total profit, while discounts of 30% and above show negative total profit in this QA summary.

Decision:
Discount analysis is feasible and should remain part of the core Phase 3 analysis.

Caution:
The discount-profit relationship is observational. The data supports describing associations between discount levels and profit, but it does not prove that discounts caused losses.

Open question:
After QA 017, review whether the candidate discount tiers should be revised, especially because the 30% discount level already shows negative total profit.

In [30]:
run_sql("""
SELECT
    discount_tier,
    COUNT(*) AS row_count,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(SUM(profit) / SUM(sales), 4) AS weighted_profit_margin
FROM raw_superstore_sales
GROUP BY discount_tier
ORDER BY
    CASE discount_tier
        WHEN 'No Discount' THEN 1
        WHEN 'Low Discount' THEN 2
        WHEN 'Moderate Discount' THEN 3
        WHEN 'High Discount' THEN 4
        ELSE 5
    END;
""")

,discount_tier,row_count,total_sales,total_profit,weighted_profit_margin
0,No Discount,4798,1087908.47,320987.60,0.2951
1,Low Discount,146,81927.87,10448.17,0.1275
2,Moderate Discount,3884,867821.02,79968.03,0.0921
3,High Discount,1166,259543.49,-125006.78,-0.4816


# Phase 2 QA Result — QA 017 Discount Tier Coverage

QA 017 passed.

Results:
- No Discount: 4,798 rows, 1,087,908.47 sales, 320,987.60 profit, 29.51% weighted profit margin
- Low Discount: 146 rows, 81,927.87 sales, 10,448.17 profit, 12.75% weighted profit margin
- Moderate Discount: 3,884 rows, 867,821.02 sales, 79,968.03 profit, 9.21% weighted profit margin
- High Discount: 1,166 rows, 259,543.49 sales, -125,006.78 profit, -48.16% weighted profit margin

Interpretation:
The candidate discount tiers are analytically useful. Profitability declines as discount tier increases. High Discount line items generated 259,543.49 in sales but -125,006.78 in profit, making high-discount activity a strong candidate for profit leakage analysis.

Decision:
Keep the current candidate discount tiers for Phase 3 analysis:
- No Discount = 0%
- Low Discount = greater than 0% to 15%
- Moderate Discount = greater than 15% to 30%
- High Discount = greater than 30%

Caution:
QA 016 showed that the exact 30% discount level already has negative total profit. Phase 3 should review whether the 30% threshold should remain in Moderate Discount or be treated as a high-risk discount level.

Causal guardrail:
The data supports describing an observed association between higher discount tiers and weaker profitability. It does not prove that discounts caused losses or that removing discounts would recover profit.

In [31]:
run_sql("""
SELECT
    customer_id,
    COUNT(DISTINCT customer_name) AS distinct_customer_names
FROM raw_superstore_sales
GROUP BY customer_id
HAVING COUNT(DISTINCT customer_name) > 1;
""")

,customer_id,distinct_customer_names


# Phase 2 QA Result — QA 018 Customer ID to Customer Name Consistency

QA 018 passed.

Result:
- Customer IDs mapping to multiple customer names: 0

Interpretation:
No customer_id maps to more than one customer_name. Customer identifiers are consistent in the staged table.

Decision:
Use customer_id as the primary customer key for customer-level analysis and Pareto contribution analysis. Use customer_name as a readable display label.

Analytical implication:
Customer-level aggregation is feasible and is not weakened by customer ID/name inconsistency.

In [32]:
run_sql("""
SELECT
    customer_name,
    COUNT(DISTINCT customer_id) AS distinct_customer_ids
FROM raw_superstore_sales
GROUP BY customer_name
HAVING COUNT(DISTINCT customer_id) > 1;
""")

,customer_name,distinct_customer_ids


# Phase 2 QA Result — QA 019 Customer Name to Customer ID Consistency

QA 019 passed.

Result:
- Customer names mapping to multiple customer IDs: 0

Interpretation:
No customer_name maps to more than one customer_id. Customer identifiers and names are consistent in both directions.

Decision:
Customer-level analysis is safe. Use customer_id as the primary grouping key and customer_name as a display label.

Analytical implication:
Customer historical contribution and customer Pareto analysis are feasible. However, this should not be described as true customer lifetime value because the dataset does not include acquisition cost, retention horizon, future value, or full lifecycle fields.

In [33]:
run_sql("""
SELECT
    product_id,
    COUNT(DISTINCT product_name) AS distinct_product_names
FROM raw_superstore_sales
GROUP BY product_id
HAVING COUNT(DISTINCT product_name) > 1
ORDER BY distinct_product_names DESC, product_id;
""")

,product_id,distinct_product_names
0,FUR-BO-10002213,2
1,FUR-CH-10001146,2
2,FUR-FU-10001473,2
3,FUR-FU-10004017,2
4,FUR-FU-10004091,2
5,FUR-FU-10004270,2
6,FUR-FU-10004848,2
7,FUR-FU-10004864,2
8,OFF-AP-10000576,2
9,OFF-AR-10001149,2


# Phase 2 QA Result — QA 020 Product ID to Product Name Consistency

QA 020 returned rows.

Result:
- 32 product_id values map to more than one product_name.
- Each flagged product_id maps to 2 distinct product names.

Interpretation:
Product ID and Product Name are not perfectly one-to-one in the staged dataset. This may reflect product-name edits, naming variation, reused identifiers, or sample-data imperfections.

Decision:
Use product_id as the product-level grouping key. Use product_name as a readable display label only.

Analytical implication:
Category and sub-category should be the primary product-mix storytelling levels. Product-level ranking can be included only with caveats. Avoid relying on product_name alone as a unique product identifier.

In [39]:
run_sql("""
SELECT
    product_name,
    COUNT(DISTINCT product_id) AS distinct_product_ids
FROM raw_superstore_sales
GROUP BY product_name
HAVING COUNT(DISTINCT product_id) > 1
ORDER BY distinct_product_ids DESC, product_name;
""")

,product_name,distinct_product_ids
0,Staples,10
1,Staple envelope,9
2,Easy-staple paper,8
3,Staples in misc. colors,7
4,Staple holder,3
5,Staple remover,3
6,"#10- 4 1/8"" x 9 1/2"" Recycled Envelopes",2
7,Avery Non-Stick Binders,2
8,Eldon Wave Desk Accessories,2
9,KI Adjustable-Height Table,2


# Phase 2 QA Result — QA 021 Product Name to Product ID Consistency

QA 021 returned rows.

Result:
Several product_name values map to multiple product_id values.

Examples:
- Staples maps to 10 distinct product IDs.
- Staple envelope maps to 9 distinct product IDs.
- Easy-staple paper maps to 8 distinct product IDs.
- Staples in misc. colors maps to 7 distinct product IDs.
- Staple holder maps to 3 distinct product IDs.
- Staple remover maps to 3 distinct product IDs.

Interpretation:
Product Name is not a reliable unique product key. Some product names are generic labels and map to multiple product IDs.

Decision:
Use product_id as the product-level grouping key. Use product_name only as a readable display label. Use category and sub_category as the main product-mix storytelling levels.

Analytical implication:
Product-level top/bottom rankings can be included only with caveats. Avoid relying on product_name alone as a unique product identifier.

In [41]:
run_sql("""
SELECT
    country,
    COUNT(*) AS row_count,
    COUNT(DISTINCT state) AS state_count,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM raw_superstore_sales
GROUP BY country;
""")

,country,row_count,state_count,total_sales,total_profit
0,United States,9994,49,2297200.86,286397.02


# Phase 2 QA Result — QA 022 Country Coverage

QA 022 passed.

Result:
- Country: United States
- Rows: 9,994
- States: 49
- Total Sales: 2,297,200.86
- Total Profit: 286,397.02

Interpretation:
The staged Superstore dataset is United States only. It contains records across 49 states.

Decision:
Use Region and State for geographic analysis. Do not create country-level comparisons because the dataset contains only one country.

Analytical implication:
Geographic analysis is feasible at the region and state level, but not at the country-comparison level.

In [43]:
run_sql("""
SELECT
    state,
    region,
    COUNT(*) AS row_count,
    COUNT(DISTINCT order_id) AS order_count,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM raw_superstore_sales
GROUP BY state, region
ORDER BY state;
""")

,state,region,row_count,order_count,total_sales,total_profit
0,Alabama,South,61,34,19510.64,5786.83
1,Arizona,West,224,108,35282.00,-3427.92
2,Arkansas,South,60,27,11678.13,4008.69
3,California,West,2001,1021,457687.63,76381.39
4,Colorado,West,182,79,32108.12,-6527.86
5,Connecticut,East,82,45,13384.36,3511.49
6,Delaware,East,96,44,27451.07,9977.37
7,District of Columbia,East,10,4,2865.02,1059.59
8,Florida,South,383,200,89473.71,-3399.30
9,Georgia,South,184,91,49095.84,16250.04


# Phase 2 QA Result — QA 023 State Coverage

QA 023 passed.

Result:
The staged table contains 49 state/geography rows across the United States.

Interpretation:
State-level geography is usable. Each state appears with one region in the output, so there is no obvious state-to-region duplication issue. This supports a Region → State geographic hierarchy.

Notable QA observations:
Some high-sales states are strongly profitable:
- California: 457,687.63 sales, 76,381.39 profit
- New York: 310,876.27 sales, 74,038.55 profit
- Washington: 138,641.27 sales, 33,402.65 profit

Some states have sizable sales but negative profit:
- Texas: 170,188.05 sales, -25,729.36 profit
- Ohio: 78,258.14 sales, -16,971.38 profit
- Pennsylvania: 116,511.91 sales, -15,559.96 profit
- Illinois: 80,166.10 sales, -12,607.89 profit
- North Carolina: 55,603.16 sales, -7,490.91 profit
- Colorado: 32,108.12 sales, -6,527.86 profit
- Tennessee: 30,661.87 sales, -5,341.69 profit
- Arizona: 35,282.00 sales, -3,427.92 profit
- Florida: 89,473.71 sales, -3,399.30 profit
- Oregon: 17,431.15 sales, -1,190.47 profit

Decision:
State-level analysis is feasible. Region should be the primary geographic dashboard level, with State used as a drill-down, tooltip or detail layer.

Caution:
Avoid overemphasizing very low-volume states when making commercial recommendations. State-level results should be interpreted alongside order count, sales volume and weighted margin.

In [44]:
run_sql("""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT row_id) AS distinct_row_ids,
    COUNT(DISTINCT order_id) AS distinct_orders,
    COUNT(DISTINCT customer_id) AS distinct_customers,
    COUNT(DISTINCT product_id) AS distinct_products,
    MIN(order_date) AS min_order_date,
    MAX(order_date) AS max_order_date,
    MIN(ship_date) AS min_ship_date,
    MAX(ship_date) AS max_ship_date,
    SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) AS negative_profit_rows,
    SUM(CASE WHEN sales <= 0 THEN 1 ELSE 0 END) AS sales_lte_zero_rows,
    SUM(CASE WHEN ship_date < order_date THEN 1 ELSE 0 END) AS ship_before_order_rows
FROM raw_superstore_sales;
""")

,row_count,distinct_row_ids,distinct_orders,distinct_customers,distinct_products,min_order_date,max_order_date,min_ship_date,max_ship_date,negative_profit_rows,sales_lte_zero_rows,ship_before_order_rows
0,9994,9994,5009,793,1862,2014-01-03,2017-12-30,2014-01-07,2018-01-05,1871,0,0


# Phase 2 QA Result — QA 024 Staging Summary for Documentation

QA 024 passed.

Results:
- Total rows: 9,994
- Distinct row IDs: 9,994
- Distinct orders: 5,009
- Distinct customers: 793
- Distinct products: 1,862
- Minimum Order Date: 2014-01-03
- Maximum Order Date: 2017-12-30
- Minimum Ship Date: 2014-01-07
- Maximum Ship Date: 2018-01-05
- Negative profit rows: 1,871
- Sales less than or equal to zero rows: 0
- Ship Date before Order Date rows: 0

Interpretation:
The staged Superstore table is structurally sound for Phase 3 analysis. Row IDs are unique, the row grain is confirmed as one product line item within an order, sales values are valid for denominator use, and date sequencing is valid.

Decision:
The staged table is approved for Phase 3 metric development and analysis.

Analytical implication:
The project can proceed with validated metrics for sales, profit, weighted profit margin, order count, customer count, product count, discount-tier analysis, negative-profit analysis, category/sub-category mix, region/state analysis, customer contribution, and year-over-year analysis.

# Phase 2 Validation Note — Row ID Assumption Confirmed

Before SQL QA, Row ID was treated as a likely row-level identifier based on field naming and dataset structure. This was not assumed as final.

Validation performed:
- Compared total row count to distinct Row ID count.
- Ran duplicate Row ID check.

Results:
- Total rows: 9,994
- Distinct Row IDs: 9,994
- Duplicate Row ID records: 0

Decision:
Row ID is confirmed as the unique row-level key.

Analytical implication:
Row-level metrics can use Row ID as the record identifier. However, order-level metrics must still use Order ID because the confirmed row grain is one product line item within an order, not one row per order.